In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/raw/city_day.csv')
print("Dataset Loaded Successfully!")
print("Shape:", df.shape)

df.head()

print("Columns:", df.columns.tolist())

Dataset Loaded Successfully!
Shape: (29531, 16)
Columns: ['City', 'Date', 'PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene', 'AQI', 'AQI_Bucket']


In [4]:
print("--- Null Values Count ---")
print(df.isnull().sum())
print("\n--- Null Percentage ---")
print((df.isnull().sum() / len(df) * 100).round(2))


print("--- Data Types ---")
print(df.dtypes)


print("Total Cities:", df['City'].nunique())
print("City Names:", df['City'].unique())


df['Date'] = pd.to_datetime(df['Date'])
print("Start Date:", df['Date'].min())
print("End Date:", df['Date'].max())


print("--- AQI Summary ---")
print(df['AQI'].describe())



--- Null Values Count ---
City              0
Date              0
PM2.5          4598
PM10          11140
NO             3582
NO2            3585
NOx            4185
NH3           10328
CO             2059
SO2            3854
O3             4022
Benzene        5623
Toluene        8041
Xylene        18109
AQI            4681
AQI_Bucket     4681
dtype: int64

--- Null Percentage ---
City           0.00
Date           0.00
PM2.5         15.57
PM10          37.72
NO            12.13
NO2           12.14
NOx           14.17
NH3           34.97
CO             6.97
SO2           13.05
O3            13.62
Benzene       19.04
Toluene       27.23
Xylene        61.32
AQI           15.85
AQI_Bucket    15.85
dtype: float64
--- Data Types ---
City              str
Date              str
PM2.5         float64
PM10          float64
NO            float64
NO2           float64
NOx           float64
NH3           float64
CO            float64
SO2           float64
O3            float64
Benzene       float6

In [ ]:
df.drop(columns=['Xylene'], inplace=True)
print("Dropped Xylene column")
print("New Shape:", df.shape)

✅ Dropped Xylene column
New Shape: (29531, 15)


In [ ]:
# Fill nulls using median of each city (smarter than global median)
cols_to_fill = ['PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 
                'NH3', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene']

df[cols_to_fill] = df.groupby('City')[cols_to_fill].transform(
    lambda x: x.fillna(x.median())
)

print("Filled missing pollutant values with city-wise median")
print("Remaining nulls:\n", df.isnull().sum())

✅ Filled missing pollutant values with city-wise median
Remaining nulls:
 City             0
Date             0
PM2.5            0
PM10          2009
NO               0
NO2              0
NOx           1169
NH3           2009
CO               0
SO2              0
O3             162
Benzene       2732
Toluene       4010
AQI           4681
AQI_Bucket    4681
dtype: int64


In [ ]:
# Rows where AQI is still null — we can't predict without a target
df.dropna(subset=['AQI', 'AQI_Bucket'], inplace=True)
print("Dropped rows with missing AQI")
print("Final Shape:", df.shape)

✅ Dropped rows with missing AQI
Final Shape: (24850, 15)


In [ ]:
df['Date'] = pd.to_datetime(df['Date'])
df['Year']  = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day']   = df['Date'].dt.day
df['Season'] = df['Month'].map({
    12:'Winter', 1:'Winter', 2:'Winter',
    3:'Spring',  4:'Spring', 5:'Spring',
    6:'Summer',  7:'Summer', 8:'Summer',
    9:'Monsoon', 10:'Monsoon', 11:'Monsoon'
})
print("Date features extracted")
df[['Date','Year','Month','Day','Season']].head()

✅ Date features extracted


,Date,Year,Month,Day,Season
28,2015-01-29,2015,1,29,Winter
29,2015-01-30,2015,1,30,Winter
30,2015-01-31,2015,1,31,Winter
31,2015-02-01,2015,2,1,Winter
32,2015-02-02,2015,2,2,Winter


In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['City_Encoded'] = le.fit_transform(df['City'])

# Save city encoder mapping for later use
city_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print("City Encoded")
print(city_mapping)

✅ City Encoded
{'Ahmedabad': np.int64(0), 'Aizawl': np.int64(1), 'Amaravati': np.int64(2), 'Amritsar': np.int64(3), 'Bengaluru': np.int64(4), 'Bhopal': np.int64(5), 'Brajrajnagar': np.int64(6), 'Chandigarh': np.int64(7), 'Chennai': np.int64(8), 'Coimbatore': np.int64(9), 'Delhi': np.int64(10), 'Ernakulam': np.int64(11), 'Gurugram': np.int64(12), 'Guwahati': np.int64(13), 'Hyderabad': np.int64(14), 'Jaipur': np.int64(15), 'Jorapokhar': np.int64(16), 'Kochi': np.int64(17), 'Kolkata': np.int64(18), 'Lucknow': np.int64(19), 'Mumbai': np.int64(20), 'Patna': np.int64(21), 'Shillong': np.int64(22), 'Talcher': np.int64(23), 'Thiruvananthapuram': np.int64(24), 'Visakhapatnam': np.int64(25)}


In [ ]:
season_map = {'Winter':0, 'Spring':1, 'Summer':2, 'Monsoon':3}
df['Season_Encoded'] = df['Season'].map(season_map)
print("Season Encoded")

✅ Season Encoded


In [ ]:
df.to_csv('../data/cleaned/city_day_cleaned.csv', index=False)
print("Cleaned dataset saved to data/cleaned/city_day_cleaned.csv")
print("Final Shape:", df.shape)

✅ Cleaned dataset saved to data/cleaned/city_day_cleaned.csv
Final Shape: (24850, 21)


In [14]:
# Some cities have ALL nulls for a pollutant, so city-median fill leaves NaN
# Fix: fill any remaining NaN with the overall column median

remaining_cols = ['PM10', 'NOx', 'NH3', 'O3', 'Benzene', 'Toluene']

for col in remaining_cols:
    df[col] = df[col].fillna(df[col].median())

print("✅ Filled remaining nulls with global median")
print(df.isnull().sum())

✅ Filled remaining nulls with global median
City              0
Date              0
PM2.5             0
PM10              0
NO                0
NO2               0
NOx               0
NH3               0
CO                0
SO2               0
O3                0
Benzene           0
Toluene           0
AQI               0
AQI_Bucket        0
Year              0
Month             0
Day               0
Season            0
City_Encoded      0
Season_Encoded    0
dtype: int64


In [13]:
print("--- Final Null Check ---")
print(df.isnull().sum())
print("\n--- Final Columns ---")
print(df.columns.tolist())

--- Final Null Check ---
City                 0
Date                 0
PM2.5                0
PM10              1893
NO                   0
NO2                  0
NOx                771
NH3               1334
CO                   0
SO2                  0
O3                 153
Benzene           2259
Toluene           3309
AQI                  0
AQI_Bucket           0
Year                 0
Month                0
Day                  0
Season               0
City_Encoded         0
Season_Encoded       0
dtype: int64

--- Final Columns ---
['City', 'Date', 'PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'AQI', 'AQI_Bucket', 'Year', 'Month', 'Day', 'Season', 'City_Encoded', 'Season_Encoded']
